In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
from src.utils.pipeline import load_all_snapshots
from src.features.plate_discipline import add_swing_flags
from src.features.arsenal import build_arsenal, separation, primary_fastball

df = load_all_snapshots()
f = add_swing_flags(df)
f = f[f["pitch_type"].notna()]

ars = build_arsenal(f, min_pitches=50)
pitchers = ars.index.get_level_values(0).unique()
print(f"{len(pitchers)} pitchers with a qualifying arsenal")

702 pitchers with a qualifying arsenal


In [2]:
# whiff rate per (pitcher, pitch_type)
sw = f[f["is_swing"]]
pt_whiff = (
    sw.groupby(["pitcher", "pitch_type"])["is_whiff"]
    .agg(["mean", "size"])
    .rename(columns={"mean": "whiff_pct", "size": "swings"})
)

rows = []
for pid in pitchers:
    sep = separation(ars, pid)
    if sep is None:
        continue
    sep["pitcher"] = pid
    rows.append(sep)

sep_all = pd.concat(rows, ignore_index=True).set_index(["pitcher", "pitch"])
sep_all.index.names = ["pitcher", "pitch_type"]

# fastball velocity, to control for it later
fb_velo = {}
for pid in pitchers:
    fb = primary_fastball(ars, pid)
    if fb is not None:
        fb_velo[pid] = ars.loc[(pid, fb), "velo"]
sep_all["fb_velo"] = sep_all.index.get_level_values(0).map(fb_velo)

data = sep_all.join(pt_whiff, how="inner")
data = data[data["swings"] >= 50]
print(f"{len(data)} (pitcher, pitch) pairs with 50+ swings")
print(data.groupby(level="pitch_type").size().sort_values(ascending=False).to_string())

1216 (pitcher, pitch) pairs with 50+ swings
pitch_type
SL    299
CH    216
ST    151
SI    134
FC    121
CU    117
FF     78
FS     58
KC     31
SV      9
KN      1
SC      1


In [3]:
print("=== POOLED (naive, mixes pitch types) ===")
print("velo_gap vs whiff:", round(data["velo_gap"].corr(data["whiff_pct"]), 3))
print("move_gap vs whiff:", round(data["move_gap"].corr(data["whiff_pct"]), 3))

=== POOLED (naive, mixes pitch types) ===
velo_gap vs whiff: 0.552
move_gap vs whiff: 0.309


In [4]:
print("=== WITHIN PITCH TYPE ===")
results = []
for pt, g in data.groupby(level="pitch_type"):
    if len(g) < 40:
        continue
    results.append({
        "pitch": pt,
        "n": len(g),
        "r_velo": g["velo_gap"].corr(g["whiff_pct"]),
        "r_move": g["move_gap"].corr(g["whiff_pct"]),
        "r_fb_velo": g["fb_velo"].corr(g["whiff_pct"]),
        "mean_whiff": g["whiff_pct"].mean(),
    })

res = pd.DataFrame(results).round(3)
print(res.sort_values("n", ascending=False).to_string(index=False))

=== WITHIN PITCH TYPE ===
pitch   n  r_velo  r_move  r_fb_velo  mean_whiff
   SL 299   0.306   0.083      0.276       0.319
   CH 216   0.279   0.049      0.303       0.289
   ST 151   0.128   0.083      0.263       0.299
   SI 134  -0.058   0.113      0.173       0.117
   FC 121   0.423   0.339      0.332       0.207
   CU 117   0.070   0.029      0.198       0.294
   FF  78   0.074   0.123      0.199       0.181
   FS  58   0.498   0.310      0.126       0.333


In [5]:
print("=== is velo_gap just fastball velocity in disguise? ===")
for pt, g in data.groupby(level="pitch_type"):
    if len(g) < 40:
        continue
    print(f"{pt}: velo_gap vs fb_velo r = {g['velo_gap'].corr(g['fb_velo']):.3f}")

=== is velo_gap just fastball velocity in disguise? ===
CH: velo_gap vs fb_velo r = 0.268
CU: velo_gap vs fb_velo r = 0.304
FC: velo_gap vs fb_velo r = 0.293
FF: velo_gap vs fb_velo r = 0.544
FS: velo_gap vs fb_velo r = 0.187
SI: velo_gap vs fb_velo r = 0.598
SL: velo_gap vs fb_velo r = 0.304
ST: velo_gap vs fb_velo r = 0.467


In [6]:
def partial_corr(g, x, y, control):
    """Correlation between x and y after removing the linear effect of control."""
    sub = g[[x, y, control]].dropna()
    if len(sub) < 20:
        return np.nan
    rx = np.polyfit(sub[control], sub[x], 1)
    ry = np.polyfit(sub[control], sub[y], 1)
    res_x = sub[x] - np.polyval(rx, sub[control])
    res_y = sub[y] - np.polyval(ry, sub[control])
    return res_x.corr(res_y)

print("=== velo_gap vs whiff, controlling for fastball velocity ===")
for pt, g in data.groupby(level="pitch_type"):
    if len(g) < 40:
        continue
    raw = g["velo_gap"].corr(g["whiff_pct"])
    ctrl = partial_corr(g, "velo_gap", "whiff_pct", "fb_velo")
    print(f"{pt}: raw {raw:+.3f}  ->  controlled {ctrl:+.3f}")

=== velo_gap vs whiff, controlling for fastball velocity ===
CH: raw +0.279  ->  controlled +0.215
CU: raw +0.070  ->  controlled +0.011
FC: raw +0.423  ->  controlled +0.361
FF: raw +0.074  ->  controlled -0.042
FS: raw +0.498  ->  controlled +0.487
SI: raw -0.058  ->  controlled -0.205
SL: raw +0.306  ->  controlled +0.242
ST: raw +0.128  ->  controlled +0.006
